# LeetCode 132 — Palindrome Partitioning II (Hard) bằng Python

Notebook này giảng **từ căn bản** một bài quy hoạch động kinh điển về chuỗi đối xứng. Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCut(self, s: str) -> int:

```

---

## Sau notebook này, bạn sẽ nắm được

1. **Palindrome** (chuỗi đối xứng) là gì, và 3 cách kiểm tra: so với chuỗi đảo `t[::-1]`, **hai con trỏ** (two pointers), và **mở rộng từ tâm** (expand around center).
2. **Backtracking** (quay lui) để liệt kê *mọi* cách chia chuỗi thành các mảnh palindrome (chính là bài #131), và vì sao cách này **bùng nổ** theo `2^(n-1)`.
3. **Đệ quy + memoization** (ghi nhớ kết quả bài toán con): từ hàm mũ xuống còn đa thức, và giới hạn **độ sâu đệ quy** của Python.
4. **Quy hoạch động (DP) với bảng 2 chiều** `is_pal[i][j]` + mảng `dp` — lời giải `O(n²)` cổ điển.
5. **Mở rộng từ tâm + cập nhật `dp` trực tiếp** — lời giải chính, `O(n²)` thời gian nhưng chỉ `O(n)` bộ nhớ, trace kỹ với `"aab"` và `"abacc"`.
6. Lỗi hay gặp (tham lam sai, quên tâm chẵn, điền bảng sai thứ tự...), mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

Lộ trình: `Backtracking (mũ)` → `Đệ quy + memo O(n³)` → `DP bảng O(n²), nhớ O(n²)` → `Expand around center O(n²), nhớ O(n)`.

## 1. Đề bài

Cho một chuỗi `s`. Hãy **chia** (partition) `s` thành các đoạn con sao cho **mỗi đoạn đều là palindrome**. Trả về **số nhát cắt ít nhất** (minimum cuts) cần dùng để có một cách chia như vậy.

> Gốc tiếng Anh: *Given a string `s`, partition `s` such that every substring of the partition is a palindrome. Return the minimum cuts needed for a palindrome partitioning of `s`.*

### Ví dụ trong đề

| Input `s` | Output | Giải thích |
|---|:---:|---|
| `"aab"` | `1` | Cách chia `["aa", "b"]` chỉ cần **1** nhát cắt: `aa | b` |
| `"a"` | `0` | Bản thân `"a"` đã là palindrome, không cần cắt |
| `"ab"` | `1` | Phải cắt thành `["a", "b"]` |

### Constraints (ràng buộc)

- `1 <= s.length <= 2000`
- `s` chỉ gồm chữ cái tiếng Anh viết thường.

Chú ý: `n` có thể lên tới **2000**. Thuật toán `O(n²)` khoảng 4 triệu bước là ổn; `O(n³)` khoảng 8 tỉ bước là **quá chậm** (TLE — Time Limit Exceeded); còn thuật toán hàm mũ thì không bao giờ xong.

## 2. Hiểu đúng "nhát cắt" và "cách chia"

Một **cách chia** (partition) là việc cắt chuỗi thành các mảnh **liên tiếp**, nối lại theo thứ tự thì được đúng chuỗi ban đầu, và **không mảnh nào rỗng**.

Ví dụ `s = "aab"` có 4 cách chia (bỏ qua yêu cầu palindrome):

| Cách chia | Số mảnh | Số nhát cắt | Mọi mảnh đều palindrome? |
|---|:-:|:-:|:-:|
| `aab` | 1 | 0 | ✘ (`"aab"` đọc ngược là `"baa"`) |
| `a | ab` | 2 | 1 | ✘ (`"ab"` không đối xứng) |
| `aa | b` | 2 | 1 | ✔ |
| `a | a | b` | 3 | 2 | ✔ |

Hai cách hợp lệ là `aa | b` (1 nhát) và `a | a | b` (2 nhát) → đáp án `1`.

Quy tắc cần nhớ ngay từ đầu:

$$\text{số nhát cắt} = \text{số mảnh} - 1$$

Vì vậy "ít nhát cắt nhất" cũng chính là "ít mảnh nhất". Ta sẽ dùng qua lại hai cách nói này, chỉ cần nhớ **trừ 1** khi trả kết quả.

Một nhận xét quan trọng: **luôn có ít nhất một cách chia hợp lệ** — cắt từng ký tự một (`n` mảnh, `n - 1` nhát), vì một ký tự đứng riêng luôn là palindrome. Nên đáp án luôn nằm trong khoảng `0 … n - 1`.

In [1]:
# Biểu diễn một cách chia bằng list các mảnh; nối lại phải ra chuỗi gốc
s = "aab"
partitions = [["aab"], ["a", "ab"], ["aa", "b"], ["a", "a", "b"]]

for p in partitions:
    assert "".join(p) == s                     # nối lại đúng chuỗi gốc
    all_pal = all(piece == piece[::-1] for piece in p)
    print(f"{' | '.join(p):<9}  mảnh={len(p)}  nhát cắt={len(p) - 1}  hợp lệ={all_pal}")

aab        mảnh=1  nhát cắt=0  hợp lệ=False
a | ab     mảnh=2  nhát cắt=1  hợp lệ=False
aa | b     mảnh=2  nhát cắt=1  hợp lệ=True
a | a | b  mảnh=3  nhát cắt=2  hợp lệ=True


## 3. Kiến thức nền 1 — Palindrome và cách kiểm tra

**Palindrome** là chuỗi đọc xuôi hay đọc ngược đều giống nhau: `"a"`, `"aa"`, `"aba"`, `"abba"`, `"racecar"`.

### Cách A: so với chuỗi đảo ngược

Trong Python, `t[::-1]` tạo chuỗi đảo ngược của `t` (cú pháp cắt `t[start:end:step]` với `step = -1` nghĩa là đi lùi). Palindrome ⇔ `t == t[::-1]`.

### Cắt đoạn `s[i:j+1]` — chú ý chỉ số

Trong bài này ta hay nói "đoạn từ vị trí `i` **đến** `j`" (tính cả hai đầu). Trong Python, `s[i:j]` **không lấy** `s[j]`, nên đoạn `i..j` phải viết là `s[i:j+1]`.

Ví dụ `s = "abacc"` (chỉ số `0 1 2 3 4`): đoạn `0..2` là `s[0:3] = "aba"`; đoạn `3..4` là `s[3:5] = "cc"`.

In [2]:
s = "abacc"
print("s[0:3] =", s[0:3], "-> đoạn 0..2")
print("s[3:5] =", s[3:5], "-> đoạn 3..4")
print("s[0:2] =", s[0:2], "-> (thiếu s[2]!) đây là đoạn 0..1")

def is_pal_reverse(t: str) -> bool:
    return t == t[::-1]

for t in ["a", "aa", "ab", "aba", "abba", "abca"]:
    print(f"{t!r:7} đảo ngược = {t[::-1]!r:7} -> {is_pal_reverse(t)}")

s[0:3] = aba -> đoạn 0..2
s[3:5] = cc -> đoạn 3..4
s[0:2] = ab -> (thiếu s[2]!) đây là đoạn 0..1
'a'     đảo ngược = 'a'     -> True
'aa'    đảo ngược = 'aa'    -> True
'ab'    đảo ngược = 'ba'    -> False
'aba'   đảo ngược = 'aba'   -> True
'abba'  đảo ngược = 'abba'  -> True
'abca'  đảo ngược = 'acba'  -> False


### Cách B: hai con trỏ (two pointers)

Cách A phải **tạo chuỗi mới** (tốn bộ nhớ và thời gian sao chép). Cách tự nhiên hơn khi kiểm tra đoạn `s[i..j]` trong chuỗi lớn: đặt một con trỏ `l` ở đầu, một con trỏ `r` ở cuối, so `s[l]` với `s[r]`; nếu bằng nhau thì **cùng tiến vào trong** (`l += 1`, `r -= 1`); gặp một cặp khác nhau là kết luận ngay "không phải palindrome".

Ví dụ `"abca"`: so `a`–`a` ✔, rồi `b`–`c` ✘ → dừng, `False`. Chỉ cần tối đa `len / 2` lần so sánh.

In [3]:
def is_pal_two_pointers(s: str, i: int, j: int, verbose: bool = False) -> bool:
    """Kiểm tra đoạn s[i..j] (tính cả hai đầu) có phải palindrome không."""
    l, r = i, j
    while l < r:
        if verbose:
            print(f"  so s[{l}]={s[l]!r} với s[{r}]={s[r]!r}")
        if s[l] != s[r]:
            return False
        l += 1
        r -= 1
    return True

print("abba:", is_pal_two_pointers("abba", 0, 3, verbose=True))
print("abca:", is_pal_two_pointers("abca", 0, 3, verbose=True))
print("đoạn 1..3 của 'xabay' ('aba'):", is_pal_two_pointers("xabay", 1, 3))

  so s[0]='a' với s[3]='a'
  so s[1]='b' với s[2]='b'
abba: True
  so s[0]='a' với s[3]='a'
  so s[1]='b' với s[2]='c'
abca: False
đoạn 1..3 của 'xabay' ('aba'): True


### Có bao nhiêu đoạn con, bao nhiêu đoạn palindrome?

Chuỗi độ dài `n` có `n(n+1)/2` đoạn con khác rỗng (chọn `i ≤ j`). Với `n = 2000` là khoảng **2 triệu** đoạn. Nếu kiểm tra mỗi đoạn mất `O(n)` thì tổng cộng là `O(n³)` — đây là gốc rễ của sự chậm mà ta sẽ phải khắc phục.

Cell dưới liệt kê mọi đoạn palindrome của `"aab"` và `"abacc"` — đó chính là "nguyên liệu" để ghép thành cách chia.

In [4]:
def palindromic_substrings(s: str):
    n = len(s)
    out = []
    for i in range(n):
        for j in range(i, n):
            if is_pal_two_pointers(s, i, j):
                out.append((i, j, s[i:j + 1]))
    return out

for s in ["aab", "abacc"]:
    n = len(s)
    pals = palindromic_substrings(s)
    print(f"{s!r}: {n * (n + 1) // 2} đoạn con, {len(pals)} đoạn palindrome:")
    print("   ", [f"{t}({i}..{j})" for i, j, t in pals])

'aab': 6 đoạn con, 4 đoạn palindrome:
    ['a(0..0)', 'aa(0..1)', 'a(1..1)', 'b(2..2)']
'abacc': 15 đoạn con, 7 đoạn palindrome:
    ['a(0..0)', 'aba(0..2)', 'b(1..1)', 'a(2..2)', 'c(3..3)', 'cc(3..4)', 'c(4..4)']


## 4. Cách 1 — Backtracking: liệt kê mọi cách chia palindrome rồi lấy min

### Kiến thức nền: liệt kê mọi cách cắt một chuỗi

Chuỗi độ dài `n` có `n - 1` "khe" giữa các ký tự. Ở mỗi khe ta **cắt** hoặc **không cắt** → có `2^(n-1)` cách chia. Ví dụ `"abc"` có 2 khe → 4 cách: `abc`, `a|bc`, `ab|c`, `a|b|c`.

Cách liệt kê tự nhiên bằng **đệ quy** (hàm tự gọi lại chính nó): "chọn mảnh **đầu tiên** `s[start:end]`, rồi giải tiếp phần còn lại `s[end:]` bằng cùng một hàm". Khi `start == len(s)` (không còn gì để chia) thì ta có một cách chia hoàn chỉnh.

**Backtracking** (quay lui) là kỹ thuật đệ quy theo nhịp 3 bước:

1. **Chọn** (choose): thêm mảnh vào danh sách `path` đang xây dựng.
2. **Khám phá** (explore): gọi đệ quy cho phần còn lại.
3. **Bỏ chọn** (unchoose): `path.pop()` để trả trạng thái lại như cũ, thử lựa chọn tiếp theo.

In [5]:
def all_splits(s: str):
    result, path = [], []

    def go(start):
        if start == len(s):                 # đã chia hết chuỗi
            result.append(path.copy())      # copy! vì path còn bị sửa tiếp
            return
        for end in range(start + 1, len(s) + 1):
            path.append(s[start:end])       # 1. chọn mảnh đầu s[start:end]
            go(end)                         # 2. khám phá phần còn lại
            path.pop()                      # 3. bỏ chọn

    go(0)
    return result

for p in all_splits("abc"):
    print(" | ".join(p))
print("Số cách chia 'abcd':", len(all_splits("abcd")), "= 2^3")

a | b | c
a | bc
ab | c
abc
Số cách chia 'abcd': 8 = 2^3


### Chỉ đi tiếp khi mảnh đầu là palindrome (chính là bài #131)

Bài LeetCode #131 *Palindrome Partitioning* yêu cầu liệt kê mọi cách chia mà mọi mảnh đều palindrome. Ta chỉ cần thêm một điều kiện vào hàm trên: **chỉ chọn mảnh `s[start:end]` nếu nó là palindrome**. Những nhánh có mảnh không đối xứng bị **cắt tỉa** (prune) ngay, không đi sâu.

Sau khi có danh sách mọi cách chia hợp lệ, đáp án bài #132 là `min(len(p) for p in partitions) - 1`.

In [6]:
def palindrome_partitions(s: str):
    result, path = [], []

    def go(start):
        if start == len(s):
            result.append(path.copy())
            return
        for end in range(start, len(s)):           # mảnh đầu là s[start..end]
            if is_pal_two_pointers(s, start, end):  # chỉ đi tiếp nếu là palindrome
                path.append(s[start:end + 1])
                go(end + 1)
                path.pop()

    go(0)
    return result

for s in ["aab", "abacc"]:
    print(f"{s!r}:")
    for p in palindrome_partitions(s):
        print(f"    {' | '.join(p):<14} nhát cắt = {len(p) - 1}")

def min_cut_backtracking(s: str) -> int:
    return min(len(p) for p in palindrome_partitions(s)) - 1

print("min_cut_backtracking:", {s: min_cut_backtracking(s) for s in ["aab", "a", "ab", "abacc", "abaab"]})

'aab':
    a | a | b      nhát cắt = 2
    aa | b         nhát cắt = 1
'abacc':
    a | b | a | c | c nhát cắt = 4
    a | b | a | cc nhát cắt = 3
    aba | c | c    nhát cắt = 2
    aba | cc       nhát cắt = 1
min_cut_backtracking: {'aab': 1, 'a': 0, 'ab': 1, 'abacc': 1, 'abaab': 1}


### Vì sao backtracking bùng nổ?

Thử chuỗi toàn chữ `a`: `"aaaa…a"`. **Mọi** đoạn con đều là palindrome, nên **mọi** cách cắt đều hợp lệ → đúng `2^(n-1)` cách chia. Với `n = 2000`, con số `2^1999` có khoảng 600 chữ số — không máy nào liệt kê nổi.

Cell dưới chỉ **đếm** số cách chia (không lưu) để thấy số này nhân đôi mỗi khi thêm một ký tự.

In [7]:
def count_palindrome_partitions(s: str) -> int:
    def go(start):
        if start == len(s):
            return 1
        return sum(go(end + 1) for end in range(start, len(s))
                   if is_pal_two_pointers(s, start, end))
    return go(0)

for k in [4, 8, 12, 16]:
    print(f"'a'*{k:<2}: {count_palindrome_partitions('a' * k):>6} cách chia  (2^{k - 1} = {2 ** (k - 1)})")
print("Với n = 2000: 2^1999 có", len(str(2 ** 1999)), "chữ số -> không thể liệt kê.")

'a'*4 :      8 cách chia  (2^3 = 8)
'a'*8 :    128 cách chia  (2^7 = 128)
'a'*12:   2048 cách chia  (2^11 = 2048)
'a'*16:  32768 cách chia  (2^15 = 32768)
Với n = 2000: 2^1999 có 602 chữ số -> không thể liệt kê.


### Nút thắt: giải đi giải lại cùng một bài toán con

Nhìn cây đệ quy của `"aab"`: sau khi chọn `a | a | …` hoặc `aa | …`, **cả hai nhánh** đều phải giải lại cùng một câu hỏi *"phần còn lại `s[2:] = "b"` chia thế nào?"*. Với chuỗi dài, một hậu tố (suffix) như `s[10:]` bị giải lại hàng nghìn lần.

Quan sát then chốt: **ta không cần biết mọi cách chia**, chỉ cần biết **số nhát ít nhất**. Và số nhát ít nhất của phần còn lại `s[i:]` chỉ phụ thuộc vào **`i`**, không phụ thuộc vào việc ta đã cắt phần đầu thế nào. → Chỉ có `n` bài toán con khác nhau! Đây là cửa ngõ dẫn tới memoization.

## 5. Cách 2 — Đệ quy + memoization: `min_cut(i)`

### Đặt câu hỏi con

Gọi `min_cut(i)` = **số nhát cắt ít nhất để chia hậu tố `s[i:]`** thành các mảnh palindrome.

- Nếu bản thân `s[i:]` đã là palindrome → `min_cut(i) = 0` (không cần cắt).
- Ngược lại, ta thử mọi **mảnh đầu** `s[i..j]` là palindrome (với `j < n - 1`, tức là còn phần phía sau), cắt **một nhát** sau `j`, rồi phần còn lại tốn `min_cut(j + 1)` nhát:

$$\text{min\_cut}(i) = \min_{\substack{i \le j < n-1 \\ s[i..j] \text{ palindrome}}} \big(1 + \text{min\_cut}(j+1)\big)$$

Đáp án bài toán là `min_cut(0)`.

Ví dụ `s = "aab"`: `min_cut(2)`: `"b"` là palindrome → `0`. `min_cut(1)`: `"ab"` không đối xứng; mảnh đầu `"a"` → `1 + min_cut(2) = 1`. `min_cut(0)`: `"aab"` không đối xứng; thử `"a"` → `1 + min_cut(1) = 2`, thử `"aa"` → `1 + min_cut(2) = 1` → lấy `1`. ✔

### Kiến thức nền: memoization là gì?

**Memoization** (ghi nhớ) = lưu kết quả của mỗi bài toán con vào một `dict` (từ điển) ngay lần đầu tính; lần sau gặp lại thì **tra ra luôn** thay vì tính lại.

Ví dụ mini kinh điển: Fibonacci `fib(n) = fib(n-1) + fib(n-2)`. Bản đệ quy thô gọi `fib(2)` rất nhiều lần; bản có `memo` mỗi `n` chỉ tính đúng một lần. Ta đếm số lần gọi hàm để thấy khác biệt.

In [8]:
calls = 0
def fib_naive(n):
    global calls
    calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

calls_memo = 0
memo_fib = {}                     # memo_fib[n] = fib(n) đã tính
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    if n in memo_fib:             # đã tính rồi -> tra ra luôn
        return memo_fib[n]
    memo_fib[n] = n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
    return memo_fib[n]

print("fib(25) =", fib_naive(25), "| số lần gọi bản thô:", calls)
print("fib(25) =", fib_memo(25), "| số lần gọi bản memo:", calls_memo)

fib(25) = 75025 | số lần gọi bản thô: 242785
fib(25) = 75025 | số lần gọi bản memo: 49


### Đệ quy `min_cut(i)` không memo vẫn bùng nổ

Viết đúng công thức trên nhưng **chưa** ghi nhớ, rồi đếm số lần gọi hàm với chuỗi `"a…ab"` (`k` chữ `a` rồi một chữ `b`). Chuỗi này đủ "khó": cả chuỗi không đối xứng (nên không dừng sớm được), nhưng mọi đoạn toàn `a` đều đối xứng (nên có rất nhiều nhánh).

In [9]:
def min_cut_recursive_count(s: str):
    n = len(s)
    calls = 0

    def solve(i):
        nonlocal calls
        calls += 1
        if is_pal_two_pointers(s, i, n - 1):   # cả s[i:] là palindrome
            return 0
        best = n - 1 - i                        # tệ nhất: cắt từng ký tự
        for j in range(i, n - 1):
            if is_pal_two_pointers(s, i, j):
                best = min(best, 1 + solve(j + 1))
        return best

    return solve(0), calls

for k in [4, 8, 12, 16]:
    ans, calls = min_cut_recursive_count("a" * k + "b")
    print(f"k={k:<2} đáp án={ans}  số lần gọi solve = {calls}")

k=4  đáp án=1  số lần gọi solve = 16
k=8  đáp án=1  số lần gọi solve = 256
k=12 đáp án=1  số lần gọi solve = 4096
k=16 đáp án=1  số lần gọi solve = 65536


Số lần gọi đúng bằng `2^k` — vẫn hàm mũ. Nhưng chỉ có `n + 1` giá trị `i` khác nhau! Thêm `memo` là đủ để mỗi `i` chỉ tính một lần.

### Code đệ quy + memo

In [10]:
def min_cut_memo(s: str) -> int:
    n = len(s)
    memo = {}                     # memo[i] = min_cut(i)

    def solve(i):
        if i in memo:
            return memo[i]
        if is_pal_two_pointers(s, i, n - 1):
            memo[i] = 0
            return 0
        best = n - 1 - i
        for j in range(i, n - 1):
            if is_pal_two_pointers(s, i, j):
                best = min(best, 1 + solve(j + 1))
        memo[i] = best
        return best

    return solve(0)

for s in ["aab", "a", "ab", "abacc", "abaab", "a" * 16 + "b"]:
    print(f"{s!r:20} -> {min_cut_memo(s)}")

'aab'                -> 1
'a'                  -> 0
'ab'                 -> 1
'abacc'              -> 1
'abaab'              -> 1
'aaaaaaaaaaaaaaaab'  -> 1


### Trace cây đệ quy với `"aab"`

Hàm `trace_memo` in mỗi lời gọi `solve(i)` (thụt lề theo độ sâu), mảnh đầu đang thử, và chỗ nào **tra được memo** thay vì tính lại. Để ý dòng `memo hit` ở `solve(2)` lần thứ hai: đó chính là công việc mà backtracking phải làm lại.

In [11]:
def trace_memo(s: str) -> int:
    n = len(s)
    memo = {}

    def solve(i, depth):
        pad = "    " * depth
        if i in memo:
            print(f"{pad}solve({i}) -> memo hit = {memo[i]}")
            return memo[i]
        if is_pal_two_pointers(s, i, n - 1):
            print(f"{pad}solve({i}): {s[i:]!r} là palindrome -> 0")
            memo[i] = 0
            return 0
        print(f"{pad}solve({i}): {s[i:]!r} không đối xứng, thử mảnh đầu")
        best = n - 1 - i
        for j in range(i, n - 1):
            if is_pal_two_pointers(s, i, j):
                print(f"{pad}  mảnh {s[i:j + 1]!r} | phần còn lại từ {j + 1}")
                best = min(best, 1 + solve(j + 1, depth + 1))
        memo[i] = best
        print(f"{pad}=> solve({i}) = {best}")
        return best

    return solve(0, 0)

print("Đáp án:", trace_memo("aab"))

solve(0): 'aab' không đối xứng, thử mảnh đầu
  mảnh 'a' | phần còn lại từ 1
    solve(1): 'ab' không đối xứng, thử mảnh đầu
      mảnh 'a' | phần còn lại từ 2
        solve(2): 'b' là palindrome -> 0
    => solve(1) = 1
  mảnh 'aa' | phần còn lại từ 2
    solve(2) -> memo hit = 0
=> solve(0) = 1
Đáp án: 1


### Độ phức tạp và hai nút thắt của cách memo

- Có `n` trạng thái `i`; mỗi trạng thái thử `O(n)` giá trị `j`; mỗi lần thử gọi `is_pal_two_pointers` tốn `O(n)`. → **Thời gian `O(n³)`**, bộ nhớ `O(n)` cho memo + ngăn xếp đệ quy.
- Với `n = 2000`: `n³ = 8 · 10⁹` → TLE chắc chắn.

**Nút thắt 1 — kiểm tra palindrome lặp lại:** đoạn `s[3..10]` có thể bị kiểm tra nhiều lần từ nhiều trạng thái khác nhau, mỗi lần tốn `O(n)`. Nếu biết trước câu trả lời cho mọi đoạn trong `O(1)` thì cả thuật toán còn `O(n²)`.

**Nút thắt 2 — độ sâu đệ quy:** `solve(0)` gọi `solve(1)` gọi `solve(2)`… độ sâu tới `n`. Python mặc định chỉ cho khoảng 1000 tầng (`sys.getrecursionlimit()`), nên chuỗi dài 2000 sẽ gây `RecursionError`. Cell dưới chứng minh điều đó — đây là lý do ta chuyển sang **DP lặp** (bottom-up, dùng vòng `for` thay vì đệ quy).

In [12]:
import sys
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())
long_s = "abc" * 500               # n = 1500, không hậu tố dài nào đối xứng
try:
    min_cut_memo(long_s)            # solve(0) -> solve(1) -> ... sâu ~1500 tầng
    print("Không lỗi (bất ngờ!)")
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

Giới hạn đệ quy mặc định: 1000
RecursionError: maximum recursion depth exceeded ...


## 6. Cách 3 — DP cổ điển: bảng `is_pal[i][j]` + mảng `dp`

Ta giải hai nút thắt cùng lúc: (1) **tính trước** palindrome cho mọi đoạn, (2) dùng **vòng lặp** thay đệ quy.

### Bước 1: tính `is_pal[i][j]` bằng ý tưởng "bóc vỏ"

Quan sát: đoạn `s[i..j]` là palindrome ⇔ **hai đầu bằng nhau** `s[i] == s[j]` **và** phần ruột bên trong `s[i+1..j-1]` cũng là palindrome. Với đoạn dài 1 hoặc 2 (`j - i < 2`) thì ruột rỗng hoặc chỉ là… không có gì, chỉ cần hai đầu bằng nhau.

$$\text{is\_pal}[i][j] = \big(s[i] = s[j]\big) \ \text{and}\ \big(j - i < 2 \ \text{or}\ \text{is\_pal}[i+1][j-1]\big)$$

Ví dụ `s = "abacc"`: `is_pal[0][2]` ("aba"): `s[0] = s[2] = 'a'` ✔ và ruột `is_pal[1][1]` ("b") ✔ → `True`. `is_pal[1][3]` ("bac"): `'b' ≠ 'c'` → `False` ngay.

**Thứ tự điền bảng:** ô `[i][j]` cần ô `[i+1][j-1]` — tức là hàng **dưới** (`i + 1`). Vì vậy ta phải duyệt `i` **từ dưới lên** (`n-1` về `0`). Mỗi ô chỉ tốn `O(1)` → cả bảng `O(n²)`.

### Kiến thức nền: tạo bảng 2 chiều trong Python (và một cái bẫy)

Bảng `n × n` là một **list chứa `n` list**. Truy cập ô hàng `i` cột `j` bằng `table[i][j]`.

Cách đúng: `[[False] * n for _ in range(n)]` — mỗi vòng tạo một list hàng **mới**.

Cách sai: `[[False] * n] * n` — tạo **một** list hàng rồi lặp lại **cùng một tham chiếu** `n` lần. Sửa một ô là cả cột thay đổi theo!

In [13]:
n = 3
good = [[False] * n for _ in range(n)]
bad = [[False] * n] * n

good[0][1] = True
bad[0][1] = True
print("good:", good)
print("bad :", bad, " <- cả 3 hàng đều đổi, vì là cùng một list!")
print("bad[0] is bad[1]:", bad[0] is bad[1])

good: [[False, True, False], [False, False, False], [False, False, False]]
bad : [[False, True, False], [False, True, False], [False, True, False]]  <- cả 3 hàng đều đổi, vì là cùng một list!
bad[0] is bad[1]: True


In [14]:
def build_is_pal(s: str):
    n = len(s)
    is_pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):          # i từ dưới lên: cần hàng i+1 trước
        for j in range(i, n):
            if s[i] == s[j] and (j - i < 2 or is_pal[i + 1][j - 1]):
                is_pal[i][j] = True
    return is_pal

def show_is_pal(s: str):
    P = build_is_pal(s)
    print("     " + "  ".join(f"{j}{s[j]}" for j in range(len(s))))
    for i in range(len(s)):
        row = "  ".join(" T" if P[i][j] else (" ." if j >= i else "  ") for j in range(len(s)))
        print(f"{i}{s[i]}  {row}")

show_is_pal("abacc")     # hàng i, cột j: T nếu s[i..j] là palindrome

     0a  1b  2a  3c  4c
0a   T   .   T   .   .
1b       T   .   .   .
2a           T   .   .
3c               T   T
4c                   T


Đọc bảng: hàng `0` có `T` ở cột `0` (`"a"`) và cột `2` (`"aba"`); hàng `3` có `T` ở cột `3` (`"c"`) và cột `4` (`"cc"`). Khớp với danh sách palindrome đã liệt kê ở mục 3. Nửa dưới đường chéo (`j < i`) bỏ trống vì không có nghĩa.

### Bước 2: `dp[i]` = số nhát ít nhất cho **tiền tố** `s[0..i]`

Lần này ta đi **từ trái sang phải** (tiền tố — prefix) cho tự nhiên với vòng `for`:

- Nếu `is_pal[0][i]` (cả tiền tố đã đối xứng) → `dp[i] = 0`.
- Ngược lại, ta xét **mảnh cuối cùng** `s[j..i]` (với `1 ≤ j ≤ i`). Nếu nó là palindrome thì: chia `s[0..j-1]` tốn `dp[j-1]` nhát, cộng **1 nhát** ngăn trước `j`:

$$dp[i] = \min_{\substack{1 \le j \le i \\ \text{is\_pal}[j][i]}} \big(dp[j-1] + 1\big)$$

Đáp án là `dp[n-1]`. Đây chính là công thức memo ở mục 5 nhưng "soi gương": xét mảnh **cuối** thay vì mảnh **đầu**.

Ví dụ `s = "aab"`: `dp[0] = 0` (`"a"`); `dp[1] = 0` (`"aa"` đối xứng); `dp[2]`: `"aab"` không đối xứng; mảnh cuối `"b"` (`j=2`) → `dp[1] + 1 = 1`; `"ab"` (`j=1`) không đối xứng → `dp[2] = 1`. ✔

In [15]:
def min_cut_dp_table(s: str) -> int:
    n = len(s)
    is_pal = build_is_pal(s)
    dp = [0] * n
    for i in range(n):
        if is_pal[0][i]:
            dp[i] = 0
            continue
        best = i                       # tệ nhất: cắt từng ký tự -> i nhát
        for j in range(1, i + 1):      # mảnh cuối s[j..i]
            if is_pal[j][i]:
                best = min(best, dp[j - 1] + 1)
        dp[i] = best
    return dp[n - 1]

for s in ["aab", "a", "ab", "abacc", "abaab", "a" * 16 + "b"]:
    print(f"{s!r:20} -> {min_cut_dp_table(s)}")

'aab'                -> 1
'a'                  -> 0
'ab'                 -> 1
'abacc'              -> 1
'abaab'              -> 1
'aaaaaaaaaaaaaaaab'  -> 1


### Trace mảng `dp` với `s = "abacc"`

| i | Tiền tố `s[0..i]` | Cả tiền tố palindrome? | Mảnh cuối palindrome `s[j..i]` → `dp[j-1]+1` | `dp[i]` |
|:-:|---|:-:|---|:-:|
| 0 | `a` | ✔ | – | 0 |
| 1 | `ab` | ✘ | `b` (j=1) → `dp[0]+1 = 1` | 1 |
| 2 | `aba` | ✔ | – | 0 |
| 3 | `abac` | ✘ | `c` (j=3) → `dp[2]+1 = 1` | 1 |
| 4 | `abacc` | ✘ | `cc` (j=3) → `dp[2]+1 = 1`; `c` (j=4) → `dp[3]+1 = 2` | 1 |

Đáp án `dp[4] = 1`: cách chia `aba | cc`. Để ý ở `i = 4`, mảnh cuối **dài hơn** (`"cc"`) cho kết quả tốt hơn mảnh ngắn (`"c"`). Hàm `trace_dp_table` in đúng bảng này.

In [16]:
def trace_dp_table(s: str) -> int:
    n = len(s)
    is_pal = build_is_pal(s)
    dp = [0] * n
    for i in range(n):
        if is_pal[0][i]:
            dp[i] = 0
            print(f"i={i} {s[:i + 1]!r:8} cả tiền tố là palindrome -> dp[{i}] = 0")
            continue
        best, options = i, []
        for j in range(1, i + 1):
            if is_pal[j][i]:
                options.append(f"{s[j:i + 1]!r}(j={j})->{dp[j - 1] + 1}")
                best = min(best, dp[j - 1] + 1)
        dp[i] = best
        print(f"i={i} {s[:i + 1]!r:8} mảnh cuối: {', '.join(options)} -> dp[{i}] = {best}")
    print("dp =", dp)
    return dp[n - 1]

print("Đáp án:", trace_dp_table("abacc"))

i=0 'a'      cả tiền tố là palindrome -> dp[0] = 0
i=1 'ab'     mảnh cuối: 'b'(j=1)->1 -> dp[1] = 1
i=2 'aba'    cả tiền tố là palindrome -> dp[2] = 0
i=3 'abac'   mảnh cuối: 'c'(j=3)->1 -> dp[3] = 1
i=4 'abacc'  mảnh cuối: 'cc'(j=3)->1, 'c'(j=4)->2 -> dp[4] = 1
dp = [0, 1, 0, 1, 1]
Đáp án: 1


### Độ phức tạp của DP bảng

Gọi `n = len(s)`.

- Dựng `is_pal`: `n(n+1)/2` ô, mỗi ô `O(1)` → `O(n²)`.
- Tính `dp`: với mỗi `i` duyệt `j` từ `1` tới `i` → `O(n²)`.
- **Thời gian `O(n²)`** (khoảng 4 triệu bước với `n = 2000` — chạy được). **Bộ nhớ `O(n²)`** cho bảng `is_pal`.

**Nút thắt còn lại:** bảng `2000 × 2000` = 4 triệu ô. Trong Python mỗi ô của list là một tham chiếu 8 byte → khoảng **32 MB** chỉ cho bảng. Chạy được, nhưng nặng. Câu hỏi: *có cần nhớ cả bảng không?* Ta chỉ dùng `is_pal` để biết "đoạn `s[j..i]` có phải palindrome không" **đúng lúc cập nhật `dp`**. Nếu có cách **sinh ra từng palindrome một** và cập nhật `dp` ngay khi gặp nó, ta không cần bảng nữa. Đó là kỹ thuật **mở rộng từ tâm**.

## 7. Cách 4 — Mở rộng từ tâm (expand around center) — lời giải chính

### Kiến thức nền: mọi palindrome đều có một **tâm**

Gấp đôi một palindrome lại thì hai nửa trùng khít nhau quanh **tâm** (center):

- Palindrome **độ dài lẻ** (`"aba"`, `"racecar"`): tâm là **một ký tự** (`b`, `e`).
- Palindrome **độ dài chẵn** (`"aa"`, `"abba"`): tâm nằm **giữa hai ký tự** (giữa hai chữ `b`).

Chuỗi dài `n` có `n` tâm lẻ (mỗi ký tự) và `n - 1` tâm chẵn (mỗi khe) → tổng `2n - 1` tâm.

**Mở rộng từ tâm:** đặt `l`, `r` tại tâm (`l = r = c` cho tâm lẻ; `l = c, r = c + 1` cho tâm chẵn). Khi `s[l] == s[r]` thì `s[l..r]` là palindrome → ghi nhận, rồi **nới rộng** `l -= 1`, `r += 1`. Gặp hai ký tự khác nhau (hoặc ra ngoài chuỗi) thì dừng: mọi đoạn rộng hơn quanh tâm này chắc chắn không đối xứng (vì đã có một cặp lệch).

Đây là "hai con trỏ" của mục 3 nhưng chạy **ngược chiều**: từ trong ra ngoài thay vì từ ngoài vào trong. Lợi ích: mỗi bước nới rộng tốn `O(1)` và **tự động cho ta một palindrome mới** — không phải kiểm tra lại từ đầu.

In [17]:
def expand_demo(s: str, l: int, r: int):
    """In các palindrome thu được khi mở rộng từ tâm (l, r)."""
    found = []
    while l >= 0 and r < len(s) and s[l] == s[r]:
        found.append(f"{s[l:r + 1]}({l}..{r})")
        l -= 1
        r += 1
    reason = "ra ngoài chuỗi" if l < 0 or r >= len(s) else f"s[{l}]={s[l]!r} != s[{r}]={s[r]!r}"
    print(f"  tìm được {found}  | dừng vì {reason}")

s = "abacc"
print("Tâm lẻ tại 1 (chữ 'b'):");      expand_demo(s, 1, 1)
print("Tâm chẵn giữa 3 và 4 ('c','c'):"); expand_demo(s, 3, 4)
print("Tâm chẵn giữa 0 và 1 ('a','b'):"); expand_demo(s, 0, 1)

Tâm lẻ tại 1 (chữ 'b'):
  tìm được ['b(1..1)', 'aba(0..2)']  | dừng vì ra ngoài chuỗi
Tâm chẵn giữa 3 và 4 ('c','c'):
  tìm được ['cc(3..4)']  | dừng vì ra ngoài chuỗi
Tâm chẵn giữa 0 và 1 ('a','b'):
  tìm được []  | dừng vì s[0]='a' != s[1]='b'


### Duyệt hết `2n - 1` tâm = liệt kê **mọi** palindrome

Nếu chạy mở rộng ở mọi tâm, ta gặp **đúng một lần** mỗi đoạn palindrome (mỗi palindrome có duy nhất một tâm). Đây chính là lời giải bài LeetCode #647 *Palindromic Substrings* (đếm số đoạn palindrome). Kiểm tra với hàm brute force ở mục 3:

In [18]:
def count_palindromes_by_center(s: str) -> int:
    n, count = len(s), 0
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):     # tâm lẻ, rồi tâm chẵn
            while l >= 0 and r < n and s[l] == s[r]:
                count += 1
                l -= 1
                r += 1
    return count

for s in ["aab", "abacc", "aaaa", "abc"]:
    a, b = count_palindromes_by_center(s), len(palindromic_substrings(s))
    print(f"{s!r:8} theo tâm: {a}, brute force: {b}")
    assert a == b

'aab'    theo tâm: 4, brute force: 4
'abacc'  theo tâm: 7, brute force: 7
'aaaa'   theo tâm: 10, brute force: 10
'abc'    theo tâm: 3, brute force: 3


### Mấu chốt: gặp palindrome `s[l..r]` thì cập nhật `dp` ngay

Để công thức gọn, ta dịch chỉ số: gọi `cut[k]` = số nhát ít nhất để chia **`k` ký tự đầu tiên** `s[0:k]` (tức `cut[k]` = `dp[k-1]` của mục 6).

Mỗi khi mở rộng tìm được palindrome `s[l..r]`, ta có một ứng cử viên cho tiền tố `s[0:r+1]`: chia xong `s[0:l]` (tốn `cut[l]` nhát), rồi thêm **1 nhát** và mảnh cuối là `s[l..r]`:

$$\text{cut}[r+1] = \min\big(\text{cut}[r+1],\ \text{cut}[l] + 1\big)$$

**Mẹo lính canh (sentinel) `cut[0] = -1`:** khi `l = 0`, mảnh cuối chính là **cả tiền tố**, không cần nhát cắt nào. Đặt `cut[0] = -1` thì `cut[0] + 1 = 0` — đúng luôn, không cần `if` riêng như `is_pal[0][i]` ở mục 6.

**Khởi tạo:** `cut[k] = k - 1` (tệ nhất: cắt sau từng ký tự). Ví dụ `n = 5`: `cut = [-1, 0, 1, 2, 3, 4]`.

**Vì sao thứ tự "tâm từ trái sang phải" là đủ?** Khi xét tâm `c`, mọi palindrome tìm được có đầu trái `l ≤ c`. Giá trị `cut[l]` chỉ phụ thuộc các palindrome **kết thúc trước vị trí `l`**, mà tâm của chúng nằm bên trái `c` → đã được xử lý xong ở các vòng trước. Nên lúc ta đọc `cut[l]`, nó đã là giá trị **cuối cùng**.

In [19]:
def min_cut_expand(s: str) -> int:
    n = len(s)
    cut = [k - 1 for k in range(n + 1)]     # cut[k]: nhát ít nhất cho s[0:k]; cut[0] = -1
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):    # tâm lẻ tại c, tâm chẵn giữa c và c+1
            while l >= 0 and r < n and s[l] == s[r]:
                # s[l..r] là palindrome -> làm mảnh cuối của tiền tố s[0:r+1]
                cut[r + 1] = min(cut[r + 1], cut[l] + 1)
                l -= 1
                r += 1
    return cut[n]

for s in ["aab", "a", "ab", "abacc", "abaab", "a" * 16 + "b"]:
    print(f"{s!r:20} -> {min_cut_expand(s)}")

'aab'                -> 1
'a'                  -> 0
'ab'                 -> 1
'abacc'              -> 1
'abaab'              -> 1
'aaaaaaaaaaaaaaaab'  -> 1


### Trace với `s = "aab"`

Khởi tạo `cut = [-1, 0, 1, 2]`. Mỗi dòng là một palindrome `s[l..r]` vừa tìm được và phép cập nhật `cut[r+1] = min(cut[r+1], cut[l] + 1)`:

| Tâm `c` | Loại | `l..r` | Palindrome | `cut[l] + 1` | `cut[r+1]` cũ → mới | `cut` sau bước |
|:-:|:-:|:-:|:-:|:-:|:-:|---|
| 0 | lẻ | 0..0 | `a` | `cut[0]+1 = 0` | `cut[1]`: 0 → 0 | `[-1, 0, 1, 2]` |
| 0 | chẵn | 0..1 | `aa` | `cut[0]+1 = 0` | `cut[2]`: 1 → **0** | `[-1, 0, 0, 2]` |
| 1 | lẻ | 1..1 | `a` | `cut[1]+1 = 1` | `cut[2]`: 0 → 0 | `[-1, 0, 0, 2]` |
| 1 | chẵn | 1..2 | – (`'a' ≠ 'b'`) | – | – | – |
| 2 | lẻ | 2..2 | `b` | `cut[2]+1 = 1` | `cut[3]`: 2 → **1** | `[-1, 0, 0, 1]` |

Đáp án `cut[3] = 1`. Mấu chốt nằm ở dòng thứ 2: tâm chẵn giữa hai chữ `a` phát hiện `"aa"` với `l = 0`, nhờ lính canh `cut[0] = -1` mà `cut[2]` thành `0` (tiền tố `"aa"` không cần cắt).

In [20]:
def trace_expand(s: str) -> int:
    n = len(s)
    cut = [k - 1 for k in range(n + 1)]
    print("khởi tạo cut =", cut)
    for c in range(n):
        for kind, (l, r) in (("lẻ", (c, c)), ("chẵn", (c, c + 1))):
            while l >= 0 and r < n and s[l] == s[r]:
                old, cand = cut[r + 1], cut[l] + 1
                cut[r + 1] = min(old, cand)
                mark = "  <- cải thiện" if cand < old else ""
                print(f"  tâm {c} {kind:4} {l}..{r} {s[l:r + 1]!r:7} "
                      f"cut[{r + 1}] = min({old}, cut[{l}]+1={cand}) = {cut[r + 1]}{mark}")
                l -= 1
                r += 1
    print("cut cuối =", cut)
    return cut[n]

print("Đáp án:", trace_expand("aab"))
print()
print("Đáp án:", trace_expand("abacc"))

khởi tạo cut = [-1, 0, 1, 2]
  tâm 0 lẻ   0..0 'a'     cut[1] = min(0, cut[0]+1=0) = 0
  tâm 0 chẵn 0..1 'aa'    cut[2] = min(1, cut[0]+1=0) = 0  <- cải thiện
  tâm 1 lẻ   1..1 'a'     cut[2] = min(0, cut[1]+1=1) = 0
  tâm 2 lẻ   2..2 'b'     cut[3] = min(2, cut[2]+1=1) = 1  <- cải thiện
cut cuối = [-1, 0, 0, 1]
Đáp án: 1

khởi tạo cut = [-1, 0, 1, 2, 3, 4]
  tâm 0 lẻ   0..0 'a'     cut[1] = min(0, cut[0]+1=0) = 0
  tâm 1 lẻ   1..1 'b'     cut[2] = min(1, cut[1]+1=1) = 1
  tâm 1 lẻ   0..2 'aba'   cut[3] = min(2, cut[0]+1=0) = 0  <- cải thiện
  tâm 2 lẻ   2..2 'a'     cut[3] = min(0, cut[2]+1=2) = 0
  tâm 3 lẻ   3..3 'c'     cut[4] = min(3, cut[3]+1=1) = 1  <- cải thiện
  tâm 3 chẵn 3..4 'cc'    cut[5] = min(4, cut[3]+1=1) = 1  <- cải thiện
  tâm 4 lẻ   4..4 'c'     cut[5] = min(1, cut[4]+1=2) = 1
cut cuối = [-1, 0, 1, 0, 1, 1]
Đáp án: 1


### Trace với `s = "abacc"` (có palindrome lẻ dài và palindrome chẵn)

Output thứ hai của cell trên. Khởi tạo `cut = [-1, 0, 1, 2, 3, 4]`.

| Tâm `c` | Loại | `l..r` | Palindrome | Cập nhật | Ghi chú |
|:-:|:-:|:-:|:-:|---|---|
| 0 | lẻ | 0..0 | `a` | `cut[1] = min(0, -1+1) = 0` | |
| 1 | lẻ | 1..1 | `b` | `cut[2] = min(1, 0+1) = 1` | |
| 1 | lẻ | 0..2 | `aba` | `cut[3] = min(2, -1+1) = 0` ✔ | nới rộng từ `b` ra `aba`, chạm `l = 0` |
| 2 | lẻ | 2..2 | `a` | `cut[3] = min(0, 1+1) = 0` | không tốt hơn |
| 3 | lẻ | 3..3 | `c` | `cut[4] = min(3, 0+1) = 1` ✔ | |
| 3 | chẵn | 3..4 | `cc` | `cut[5] = min(4, 0+1) = 1` ✔ | tâm chẵn giữa hai `c` |
| 4 | lẻ | 4..4 | `c` | `cut[5] = min(1, 1+1) = 1` | mảnh `"c"` đơn lẻ kém hơn `"cc"` |

Các tâm chẵn còn lại (`a|b`, `b|a`, `a|c`) dừng ngay vì hai ký tự khác nhau. Kết quả `cut = [-1, 0, 1, 0, 1, 1]`, đáp án `1` (`aba | cc`) — trùng với DP bảng ở mục 6, nhưng **không cần bảng `n × n`**.

### Vì sao đúng? (invariant bằng lời)

**Bất biến (invariant):** *ngay trước khi xử lý tâm `c`, các giá trị `cut[0], cut[1], …, cut[c]` đã đúng là số nhát ít nhất cho các tiền tố tương ứng.*

- **Không bỏ sót:** cách chia tối ưu của tiền tố `s[0:k]` có một mảnh cuối `s[l..k-1]` là palindrome. Palindrome này có một tâm, và khi vòng lặp tới tâm đó, việc mở rộng **chắc chắn** đi qua đúng đoạn `l..k-1` (mở rộng chỉ dừng khi gặp cặp lệch, mà bên trong một palindrome không có cặp lệch nào). Lúc đó ta thực hiện `cut[k] = min(cut[k], cut[l] + 1)`, và `cut[l]` đã đúng nhờ bất biến (vì `l ≤ c`). Nên `cut[k]` đạt được giá trị tối ưu.
- **Không trả nhầm:** mỗi lần cập nhật đều ứng với một cách chia **thật sự** hợp lệ (cách chia tốt nhất của `s[0:l]` + mảnh palindrome `s[l..r]`), nên `cut[k]` không bao giờ nhỏ hơn giá trị tối ưu.
- **Giữ bất biến:** sau khi xử lý tâm `c`, mọi palindrome kết thúc tại vị trí `c` (tức ảnh hưởng `cut[c+1]`) đều có tâm `≤ c` → đã được xét hết → `cut[c+1]` đã đúng.

### Độ phức tạp

Gọi `n = len(s)`.

- **Thời gian `O(n²)`:** có `2n - 1` tâm, mỗi tâm mở rộng tối đa khoảng `n/2` bước. Trường hợp tệ nhất là `"aaa…a"` (mọi đoạn đều đối xứng, mở rộng tới tận biên), khoảng `n²/2` bước. Trường hợp tốt (chuỗi ít palindrome như `"abcd…"`) mỗi tâm dừng ngay → gần `O(n)`.
- **Bộ nhớ `O(n)`:** chỉ mảng `cut` dài `n + 1`. Không bảng 2 chiều, không đệ quy.

## 8. Code nộp LeetCode

Bản chính `Solution` dùng **mở rộng từ tâm** — ngắn, `O(n²)` thời gian, `O(n)` bộ nhớ, không đệ quy. Một mẹo nhỏ để chạy nhanh hơn trong Python: dùng `if` thay cho hàm `min()` (gọi hàm tốn hơn một phép so sánh).

**Bản thay thế `SolutionDP`: DP bảng** (dễ giải thích từng bước khi phỏng vấn). Ta gộp hai vòng thành một: khi xét `i` (đầu phải), duyệt `j` từ `0` tới `i` và tính `is_pal[j][i]` ngay — ô `[j+1][i-1]` thuộc cột `i - 1` đã tính xong ở vòng trước. Dùng `cut` với lính canh `-1` để bỏ trường hợp đặc biệt. (Trên LeetCode nhớ đổi tên `SolutionDP` thành `Solution`.)

In [21]:
class Solution:
    def minCut(self, s: str) -> int:
        n = len(s)
        cut = [k - 1 for k in range(n + 1)]   # cut[k]: số nhát ít nhất cho s[0:k]
        for c in range(n):
            for l, r in ((c, c), (c, c + 1)):  # tâm lẻ, tâm chẵn
                while l >= 0 and r < n and s[l] == s[r]:
                    if cut[l] + 1 < cut[r + 1]:
                        cut[r + 1] = cut[l] + 1
                    l -= 1
                    r += 1
        return cut[n]


class SolutionDP:
    def minCut(self, s: str) -> int:
        n = len(s)
        is_pal = [[False] * n for _ in range(n)]
        cut = [k - 1 for k in range(n + 1)]
        for i in range(n):                    # i: đầu phải của mảnh cuối
            for j in range(i + 1):            # j: đầu trái
                if s[j] == s[i] and (i - j < 2 or is_pal[j + 1][i - 1]):
                    is_pal[j][i] = True
                    if cut[j] + 1 < cut[i + 1]:
                        cut[i + 1] = cut[j] + 1
        return cut[n]

### Bộ test: ví dụ đề + edge case

| Nhóm | Input | Expected | Vì sao |
|---|---|:-:|---|
| Ví dụ đề | `"aab"`, `"a"`, `"ab"` | 1, 0, 1 | |
| Cả chuỗi đối xứng | `"racecar"`, `"abba"`, `"aaaa"` | 0, 0, 0 | không cần cắt |
| Toàn ký tự khác nhau | `"abcde"` | 4 | phải cắt từng ký tự |
| Palindrome lẻ + chẵn | `"abacc"` | 1 | `aba | cc` |
| Tham lam sai | `"abaab"` | 1 | `a | baab` (không phải `aba | a | b`) |
| Palindrome ở giữa | `"cabababcbc"` | 3 | |
| Hai khối | `"aabb"` | 1 | `aa | bb` |
| Biên trên `n = 2000` | `"a" * 2000`, `"ab" * 1000` | 0, 1 | tệ nhất cho mở rộng từ tâm |

In [22]:
tests = [
    ("aab", 1), ("a", 0), ("ab", 1),
    ("racecar", 0), ("abba", 0), ("aaaa", 0),
    ("abcde", 4),
    ("abacc", 1),
    ("abaab", 1),
    ("cabababcbc", 3),
    ("aabb", 1),
]
for cls in (Solution, SolutionDP):
    sol = cls()
    for s, expected in tests:
        got = sol.minCut(s)
        assert got == expected, (cls.__name__, s, got, expected)
    print(f"{cls.__name__}: tất cả {len(tests)} test PASS!")

import time
for big, expected in [("a" * 2000, 0), ("ab" * 1000, 1)]:
    t0 = time.perf_counter()
    assert Solution().minCut(big) == expected
    print(f"n=2000 {big[:6]!r}... PASS, đáp án {expected} ({time.perf_counter() - t0:.2f} giây)")

Solution: tất cả 11 test PASS!
SolutionDP: tất cả 11 test PASS!


n=2000 'aaaaaa'... PASS, đáp án 0 (0.18 giây)


n=2000 'ababab'... PASS, đáp án 1 (0.09 giây)


## 9. Đối chiếu chéo các cách trên input ngẫu nhiên

Backtracking chậm nhưng đúng **theo định nghĩa** (xét hết mọi cách chia), nên ta dùng nó làm "trọng tài" cho các cách còn lại trên chuỗi ngắn. Dùng bảng chữ cái nhỏ `"ab"` / `"abc"` để chuỗi có nhiều palindrome (nhiều nhánh đặc biệt). `random.seed(132)` cố định để lần chạy nào cũng ra cùng dữ liệu.

In [23]:
import random
random.seed(132)

methods = {
    "memo": min_cut_memo,
    "dp_table": min_cut_dp_table,
    "expand": min_cut_expand,
    "Solution": Solution().minCut,
    "SolutionDP": SolutionDP().minCut,
}
for _ in range(1500):
    s = "".join(random.choice(random.choice(["ab", "abc"])) for _ in range(random.randint(1, 11)))
    expected = min_cut_backtracking(s)
    for name, f in methods.items():
        assert f(s) == expected, (name, s, f(s), expected)
print("1500 chuỗi ngắn: mọi cách khớp backtracking!")

for _ in range(200):     # chuỗi dài hơn: so các cách nhanh với nhau
    s = "".join(random.choice("ab") for _ in range(random.randint(50, 300)))
    results = {name: f(s) for name, f in methods.items() if name != "memo"}
    assert len(set(results.values())) == 1, (s, results)
print("200 chuỗi dài 50..300: dp_table, expand, Solution, SolutionDP khớp nhau!")

1500 chuỗi ngắn: mọi cách khớp backtracking!


200 chuỗi dài 50..300: dp_table, expand, Solution, SolutionDP khớp nhau!


## 10. So sánh các cách

Gọi `n = len(s)`.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|:-:|:-:|---|
| Backtracking (#131) | liệt kê mọi cách chia palindrome, lấy ít mảnh nhất | `O(n · 2^n)` | `O(n)` (+ danh sách kết quả) | dễ nghĩ, dùng làm "trọng tài" test |
| Đệ quy + memo | `min_cut(i)` cho hậu tố, thử mọi mảnh đầu | `O(n³)` | `O(n)` | đúng hướng DP, nhưng kiểm tra palindrome lặp + `RecursionError` |
| DP bảng | `is_pal[i][j]` "bóc vỏ" + `dp` tiền tố | `O(n²)` | `O(n²)` | cổ điển, dễ trình bày, tốn ~32 MB với `n = 2000` |
| **Mở rộng từ tâm** | sinh từng palindrome, cập nhật `cut[r+1]` ngay | `O(n²)` | `O(n)` | **nên dùng**: ngắn, nhanh, ít bộ nhớ |

Mạch cải tiến: *liệt kê hết* → *nhận ra bài toán con chỉ phụ thuộc `i`* (memo) → *tính trước palindrome + bỏ đệ quy* (bảng) → *không cần nhớ bảng vì có thể sinh palindrome đúng lúc cần* (tâm).

## 11. Lỗi thường gặp

### Lỗi 1: Tham lam — luôn lấy palindrome **dài nhất** ở đầu

Trực giác "mảnh càng dài thì càng ít nhát cắt" nghe hợp lý nhưng **sai**. Với `s = "abaab"`: tham lam lấy `"aba"` (palindrome đầu dài nhất), phần còn lại `"ab"` phải cắt thành `a | b` → tổng `aba | a | b` = **2** nhát. Nhưng tối ưu là `a | baab` = **1** nhát. Chọn mảnh đầu ngắn hơn lại mở ra một mảnh sau dài hơn — đó là lý do phải dùng DP xét **mọi** lựa chọn.

In [24]:
def wrong_greedy(s: str) -> int:
    cuts, i, n = 0, 0, len(s)
    while i < n:
        j = max(j for j in range(i, n) if is_pal_two_pointers(s, i, j))  # mảnh dài nhất từ i
        i = j + 1
        if i < n:
            cuts += 1
    return cuts

for s in ["aab", "abaab", "abacc"]:
    print(f"{s!r:8} greedy = {wrong_greedy(s)}, đúng = {min_cut_expand(s)}")

'aab'    greedy = 1, đúng = 1
'abaab'  greedy = 2, đúng = 1
'abacc'  greedy = 1, đúng = 1


### Lỗi 2: Chỉ mở rộng từ tâm **lẻ**, quên tâm chẵn

Nếu chỉ chạy `l = r = c`, ta chỉ tìm được palindrome độ dài lẻ và bỏ lỡ `"aa"`, `"abba"`, `"cc"`… Ngay ví dụ đầu tiên của đề đã sai: `"aab"` cho `2` thay vì `1`.

In [25]:
def wrong_odd_only(s: str) -> int:
    n = len(s)
    cut = [k - 1 for k in range(n + 1)]
    for c in range(n):
        l, r = c, c                        # THIẾU tâm chẵn (c, c + 1)
        while l >= 0 and r < n and s[l] == s[r]:
            cut[r + 1] = min(cut[r + 1], cut[l] + 1)
            l -= 1
            r += 1
    return cut[n]

for s in ["aab", "abba", "abacc"]:
    print(f"{s!r:8} chỉ tâm lẻ = {wrong_odd_only(s)}, đúng = {min_cut_expand(s)}")

'aab'    chỉ tâm lẻ = 2, đúng = 1
'abba'   chỉ tâm lẻ = 3, đúng = 0
'abacc'  chỉ tâm lẻ = 2, đúng = 1


### Lỗi 3: Điền bảng `is_pal` sai thứ tự (`i` tăng dần)

Ô `is_pal[i][j]` cần ô `is_pal[i+1][j-1]` ở **hàng dưới**. Nếu duyệt `i` từ `0` lên, lúc cần đọc hàng `i + 1` thì nó **chưa được tính** (vẫn là `False`) → mọi palindrome dài từ 3 trở lên bị bỏ sót. Với `"aba"`, ô `is_pal[0][2]` thành `False`, nên thuật toán sẽ cắt `a | b | a` (2 nhát) thay vì đáp án đúng `0`.

Sửa: duyệt `i` **giảm dần** (`range(n-1, -1, -1)`), hoặc như `SolutionDP` — duyệt theo cột phải `i` tăng dần (cột `i - 1` đã xong).

In [26]:
def wrong_build_is_pal(s: str):
    n = len(s)
    P = [[False] * n for _ in range(n)]
    for i in range(n):                     # SAI: i tăng dần, hàng i+1 chưa tính
        for j in range(i, n):
            P[i][j] = s[i] == s[j] and (j - i < 2 or P[i + 1][j - 1])
    return P

for s in ["aba", "abacc"]:
    P_wrong, P_right = wrong_build_is_pal(s), build_is_pal(s)
    print(f"{s!r}: is_pal[0][2] sai = {P_wrong[0][2]}, đúng = {P_right[0][2]}")

'aba': is_pal[0][2] sai = False, đúng = True
'abacc': is_pal[0][2] sai = False, đúng = True


### Lỗi 4: Quên lính canh `cut[0] = -1` (khởi tạo `cut[k] = k`)

Nếu khởi tạo `cut = [0, 1, 2, …, n]` thì `cut[k]` thực chất đếm **số mảnh** chứ không phải số nhát cắt — mọi kết quả bị **dư 1**. `"aab"` cho `2`, `"a"` cho `1`.

In [27]:
def wrong_no_sentinel(s: str) -> int:
    n = len(s)
    cut = list(range(n + 1))               # SAI: cut[0] = 0 thay vì -1
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):
            while l >= 0 and r < n and s[l] == s[r]:
                cut[r + 1] = min(cut[r + 1], cut[l] + 1)
                l -= 1
                r += 1
    return cut[n]

for s in ["aab", "a", "abacc"]:
    print(f"{s!r:8} không lính canh = {wrong_no_sentinel(s)}, đúng = {min_cut_expand(s)}")

'aab'    không lính canh = 2, đúng = 1
'a'      không lính canh = 1, đúng = 0
'abacc'  không lính canh = 2, đúng = 1


### Lỗi 5: Lặt vặt khác

| Lỗi | Hậu quả | Sửa |
|---|---|---|
| `[[False] * n] * n` | mọi hàng là cùng một list, sửa một ô đổi cả cột | `[[False] * n for _ in range(n)]` |
| Viết `s[i:j]` khi muốn đoạn `i..j` | thiếu ký tự `s[j]`, kiểm tra nhầm đoạn | `s[i:j+1]` |
| Đệ quy + memo với `n = 2000` | `RecursionError` (giới hạn ~1000 tầng), và `O(n³)` TLE | dùng DP lặp |
| Kiểm tra palindrome bằng `s[j:i+1] == s[j:i+1][::-1]` bên trong vòng `j` | quay lại `O(n³)` dù đã "dùng DP" | dùng bảng `is_pal` hoặc mở rộng từ tâm |
| Trả về số mảnh thay vì số nhát | dư 1 | nhát = mảnh − 1 |
| Điều kiện `while` thiếu `l >= 0` | `s[-1]` là ký tự **cuối** trong Python, không báo lỗi mà ra kết quả sai | kiểm tra biên trước `s[l] == s[r]` |

## 12. Mẹo nhận dạng pattern

- Đề có **"chia chuỗi/mảng thành các đoạn liên tiếp"** + **"ít nhất / nhiều nhất / số cách"** → nghĩ ngay tới **DP trên tiền tố**: `dp[i]` = đáp án cho `i` phần tử đầu, chuyển trạng thái bằng cách chọn **đoạn cuối** `[j..i]`. Cùng họ: #139 Word Break, #1278, #1745, #91 Decode Ways.
- Đề có **"palindrome"** + cần biết **nhiều đoạn con** có đối xứng không → hai công cụ: bảng `is_pal` "bóc vỏ" `O(n²)` hoặc **mở rộng từ tâm** (lẻ + chẵn).
- Câu "liệt kê **mọi** cách" → backtracking (#131). Câu "**ít nhất** bao nhiêu" → gần như luôn là DP, đừng liệt kê.
- Nếu DP cần tra một bảng 2 chiều nhưng chỉ tra **đúng lúc** sinh ra phần tử đó → thử **đảo vòng lặp**: sinh từng phần tử và "đẩy" (push) cập nhật vào `dp` thay vì "kéo" (pull) từ bảng. Đó là cách ta bỏ được bảng `O(n²)` ở bài này.
- Thấy `n ≤ 2000` → gần như chắc đề mong `O(n²)`.

## 13. Biến thể và bài liên quan

| Bài | Nội dung | Liên hệ |
|---|---|---|
| #131 Palindrome Partitioning (Medium) | liệt kê mọi cách chia palindrome | Cách 1 của notebook này |
| #5 Longest Palindromic Substring (Medium) | đoạn palindrome dài nhất | mở rộng từ tâm, giữ đoạn dài nhất |
| #647 Palindromic Substrings (Medium) | đếm số đoạn palindrome | `count_palindromes_by_center` ở mục 7 |
| #1278 Palindrome Partitioning III (Hard) | sửa ít ký tự nhất để chia thành đúng `k` palindrome | DP 2 chiều — Bài 6 bên dưới |
| #1745 Palindrome Partitioning IV (Hard) | có chia được thành đúng 3 palindrome không | bảng `is_pal` + thử 2 điểm cắt — Bài 5 |
| #139 Word Break (Medium) | chia chuỗi thành các từ trong từ điển | cùng khung DP tiền tố |

### Biến thể nhỏ: trả về **chính cách chia** tối ưu

Đề chỉ hỏi số nhát, nhưng trong thực tế ta thường muốn biết cắt ở đâu. Chỉ cần lưu thêm `start[k]` = đầu trái `l` của **mảnh cuối** trong cách chia tốt nhất của `s[0:k]`, rồi lần ngược từ `k = n` về `0`.

In [28]:
def min_cut_partition(s: str):
    n = len(s)
    cut = [k - 1 for k in range(n + 1)]
    start = [0] * (n + 1)                  # start[k]: đầu trái mảnh cuối của s[0:k]
    for k in range(1, n + 1):
        start[k] = k - 1                   # mặc định: mảnh cuối là 1 ký tự
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):
            while l >= 0 and r < n and s[l] == s[r]:
                if cut[l] + 1 < cut[r + 1]:
                    cut[r + 1] = cut[l] + 1
                    start[r + 1] = l
                l -= 1
                r += 1
    pieces, k = [], n
    while k > 0:                           # lần ngược
        pieces.append(s[start[k]:k])
        k = start[k]
    return cut[n], pieces[::-1]

for s in ["aab", "abacc", "abaab", "cabababcbc"]:
    cuts, pieces = min_cut_partition(s)
    assert len(pieces) - 1 == cuts and "".join(pieces) == s
    assert all(p == p[::-1] for p in pieces)
    print(f"{s!r:13} {cuts} nhát: {' | '.join(pieces)}")

'aab'         1 nhát: aa | b
'abacc'       1 nhát: aba | cc
'abaab'       1 nhát: a | baab
'cabababcbc'  3 nhát: c | a | babab | cbc


## 14. Bài tập tự luyện

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 15) và **Đáp án** (mục 16, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

`Solution().minCut` trả về gì với từng input sau? Với mỗi câu, viết luôn một cách chia tối ưu.

```python
"aaaa"
"abcde"
"abaab"
"aabba"
"banana"
```

### Bài 2 — Tính tay mảng `cut` bằng mở rộng từ tâm

Với `s = "abbab"`, khởi tạo `cut = [-1, 0, 1, 2, 3, 4]`. Lập bảng giống mục 7: mỗi dòng là một palindrome `s[l..r]` tìm được (theo thứ tự tâm `0, 1, 2, …`, tâm lẻ trước tâm chẵn) và phép cập nhật `cut[r+1]`. Ghi mảng `cut` cuối cùng và đáp án.

### Bài 3 — Điền TODO: mở rộng từ tâm

Hoàn thiện hàm theo đúng thuật toán mục 7.

In [29]:
def exercise_expand(s: str) -> int:
    n = len(s)
    # TODO 1: khởi tạo cut với lính canh cut[0] = -1 (cut[k] = k - 1)
    cut = None
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):
            # TODO 2: while còn trong biên và s[l] == s[r]:
            #             cập nhật cut[r + 1] bằng cut[l] + 1 nếu tốt hơn
            #             nới rộng l, r
            pass
    # TODO 3: trả về số nhát cho cả chuỗi
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_expand("aab") == 1
# assert exercise_expand("abacc") == 1
# assert exercise_expand("abaab") == 1
# assert exercise_expand("a") == 0

### Bài 4 — Tự viết lại: DP bảng `is_pal` + `dp`

Không nhìn lại mục 6, tự viết hàm gồm hai bước: (1) dựng bảng `is_pal` với thứ tự điền đúng, (2) tính `dp[i]` cho tiền tố `s[0..i]` (được dùng `dp` có chỉ số thường, **không** dùng lính canh — để luyện xử lý trường hợp "cả tiền tố là palindrome").

In [30]:
def exercise_dp_table(s: str) -> int:
    # TODO 1: is_pal = bảng n x n (cẩn thận bẫy [[False]*n]*n)
    # TODO 2: điền is_pal với i giảm dần
    # TODO 3: dp[i] = 0 nếu is_pal[0][i], ngược lại min(dp[j-1] + 1) với is_pal[j][i]
    raise NotImplementedError

# Gợi ý test:
# assert exercise_dp_table("aab") == 1
# assert exercise_dp_table("abacc") == 1
# assert exercise_dp_table("racecar") == 0
# assert exercise_dp_table("abcde") == 4

### Bài 5 — Biến thể (LeetCode #1745 Palindrome Partitioning IV)

Cho chuỗi `s` (`3 <= len(s) <= 2000`, chữ thường). Trả về `True` nếu có thể chia `s` thành **đúng 3** đoạn khác rỗng mà cả 3 đều là palindrome.

```python
check_partitioning("abcbdd")  -> True    # "a" | "bcb" | "dd"
check_partitioning("bcbddxy") -> False
check_partitioning("aaa")     -> True    # "a" | "a" | "a"
check_partitioning("abc")     -> True    # "a" | "b" | "c"
```

Yêu cầu `O(n²)`.

In [31]:
def check_partitioning(s: str) -> bool:
    # TODO: dựng is_pal; thử mọi cặp điểm cắt (i, j) với 0 < i < j < n
    #       sao cho s[0..i-1], s[i..j-1], s[j..n-1] đều là palindrome
    pass

# Gợi ý test:
# assert check_partitioning("abcbdd") is True
# assert check_partitioning("bcbddxy") is False
# assert check_partitioning("aaa") is True
# assert check_partitioning("abc") is True

### Bài 6 — Mini challenge (LeetCode #1278 Palindrome Partitioning III)

Cho chuỗi `s` và số nguyên `k` (`1 <= k <= len(s) <= 100`, chữ thường). Bạn được **đổi** một số ký tự của `s` thành ký tự khác, rồi chia `s` thành **đúng `k`** đoạn khác rỗng, mỗi đoạn là palindrome. Trả về **số ký tự ít nhất** cần đổi.

```python
palindrome_partition("abc", 2)      -> 1   # "ab" đổi thành "aa" (hoặc "bb"), chia "aa" | "c"
palindrome_partition("aabbc", 3)    -> 0   # "aa" | "bb" | "c"
palindrome_partition("leetcode", 8) -> 0   # mỗi ký tự một đoạn
```

Tự kiểm tra lại bằng một bản brute force (thử mọi cách đặt `k - 1` điểm cắt bằng `itertools.combinations`).

In [32]:
def palindrome_partition(s: str, k: int) -> int:
    # TODO 1: cost[i][j] = số ký tự cần đổi để s[i..j] thành palindrome
    #         (đếm số cặp lệch khi so từ hai đầu vào giữa)
    # TODO 2: f[m][i] = chi phí ít nhất để chia s[0:i] thành đúng m đoạn
    # TODO 3: f[m][i] = min(f[m-1][j] + cost[j][i-1]) với m-1 <= j < i
    pass

# Gợi ý test:
# assert palindrome_partition("abc", 2) == 1
# assert palindrome_partition("aabbc", 3) == 0
# assert palindrome_partition("leetcode", 8) == 0

## 15. Gợi ý theo mức

**Bài 1**
- Mức 1: kiểm tra trước xem cả chuỗi có đối xứng không (đáp án `0`).
- Mức 2: chuỗi không có hai ký tự liền kề hoặc cách một nào trùng nhau thì phải cắt từng ký tự.
- Mức 3: `"aabba"` — để ý `"abba"`; `"banana"` — để ý `"anana"`.

**Bài 2**
- Mức 1: đi lần lượt tâm `0..4`; mỗi tâm thử lẻ `(c, c)` rồi chẵn `(c, c+1)`.
- Mức 2: tâm chẵn giữa hai chữ `b` (vị trí 1 và 2) nới được tới `"abba"` chạm `l = 0`.
- Mức 3: ở tâm `3`, palindrome `"bab"` (2..4) cho `cut[5] = cut[2] + 1 = 2`, sau đó tâm `4` còn cải thiện tiếp.

**Bài 3**
- Mức 1: chỉ 3 chỗ cần điền, đúng như `min_cut_expand`.
- Mức 2: `cut = [k - 1 for k in range(n + 1)]`; điều kiện `while l >= 0 and r < n and s[l] == s[r]`.
- Mức 3: trong vòng `while`: cập nhật `cut[r + 1]`, rồi `l -= 1; r += 1`; cuối cùng `return cut[n]`.

**Bài 4**
- Mức 1: hai vòng `for` lồng nhau cho bảng, hai vòng cho `dp`.
- Mức 2: `for i in range(n - 1, -1, -1): for j in range(i, n):`.
- Mức 3: ở bước `dp`, `best = i` (cắt từng ký tự), duyệt `j` từ `1` tới `i`.

**Bài 5**
- Mức 1: cần biết nhanh "đoạn `a..b` có đối xứng không" cho rất nhiều đoạn → bảng `is_pal`.
- Mức 2: hai vòng lặp cho điểm cắt `i` (đầu đoạn 2) và `j` (đầu đoạn 3): `O(n²)` cặp, mỗi cặp tra 3 ô `O(1)`.
- Mức 3: `if is_pal[0][i-1] and is_pal[i][j-1] and is_pal[j][n-1]: return True`.

**Bài 6**
- Mức 1: tách làm hai bài con: "biến một đoạn thành palindrome tốn bao nhiêu?" và "chia thành đúng `k` đoạn".
- Mức 2: `cost[i][j]` = số chỉ số `t` trong nửa đầu mà `s[i+t] != s[j-t]`; có thể tính bằng công thức bóc vỏ `cost[i][j] = cost[i+1][j-1] + (s[i] != s[j])`.
- Mức 3: `f[0][0] = 0`, còn lại `inf`; đáp án `f[k][n]`. Độ phức tạp `O(k · n²)`.

## 16. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:**

| Input | Đáp án | Một cách chia tối ưu |
|---|:-:|---|
| `"aaaa"` | 0 | `aaaa` |
| `"abcde"` | 4 | `a | b | c | d | e` |
| `"abaab"` | 1 | `a | baab` |
| `"aabba"` | 1 | `a | abba` |
| `"banana"` | 1 | `b | anana` |

**Bài 2:** `s = "abbab"`, khởi tạo `cut = [-1, 0, 1, 2, 3, 4]`.

| Tâm | Loại | `l..r` | Palindrome | Cập nhật |
|:-:|:-:|:-:|:-:|---|
| 0 | lẻ | 0..0 | `a` | `cut[1] = min(0, -1+1) = 0` |
| 1 | lẻ | 1..1 | `b` | `cut[2] = min(1, 0+1) = 1` |
| 1 | chẵn | 1..2 | `bb` | `cut[3] = min(2, 0+1) = 1` |
| 1 | chẵn | 0..3 | `abba` | `cut[4] = min(3, -1+1) = 0` |
| 2 | lẻ | 2..2 | `b` | `cut[3] = min(1, 1+1) = 1` |
| 3 | lẻ | 3..3 | `a` | `cut[4] = min(0, 1+1) = 0` |
| 3 | lẻ | 2..4 | `bab` | `cut[5] = min(4, 1+1) = 2` |
| 4 | lẻ | 4..4 | `b` | `cut[5] = min(2, 0+1) = 1` |

`cut` cuối `= [-1, 0, 1, 1, 0, 1]`, đáp án `1` (`abba | b`). Để ý `"bab"` là palindrome dài nhưng **không** phải lựa chọn tốt nhất: mảnh `"b"` đơn lẻ sau `"abba"` thắng. Cell dưới kiểm chứng cả hai bài bằng code.

**Bài 5:** dựng bảng `is_pal` một lần (`O(n²)`), rồi thử mọi cặp điểm cắt `(i, j)` — mỗi cặp chỉ tra 3 ô. Kiểm chứng với backtracking (có cách chia nào gồm đúng 3 mảnh không).

**Bài 6:** `cost[i][j]` tính bằng công thức bóc vỏ giống `is_pal`: hai đầu lệch thì phải đổi 1 ký tự, cộng chi phí phần ruột. `f[m][i]` là chi phí ít nhất để chia `i` ký tự đầu thành đúng `m` đoạn; đoạn cuối là `s[j..i-1]`, phần trước `s[0:j]` phải chia thành `m - 1` đoạn (cần `j >= m - 1` để mỗi đoạn có ít nhất 1 ký tự). Thời gian `O(k · n²)`, bộ nhớ `O(n²)`. Kiểm chứng với brute force thử mọi bộ `k - 1` điểm cắt.

In [33]:
# Kiểm chứng Bài 1 và Bài 2
answers_1 = {"aaaa": 0, "abcde": 4, "abaab": 1, "aabba": 1, "banana": 1}
for s, expected in answers_1.items():
    assert Solution().minCut(s) == expected, s
    cuts, pieces = min_cut_partition(s)
    print(f"{s!r:9} -> {cuts}  ({' | '.join(pieces)})")

print()
assert trace_expand("abbab") == 1

'aaaa'    -> 0  (aaaa)
'abcde'   -> 4  (a | b | c | d | e)
'abaab'   -> 1  (a | baab)
'aabba'   -> 1  (a | abba)
'banana'  -> 1  (b | anana)

khởi tạo cut = [-1, 0, 1, 2, 3, 4]
  tâm 0 lẻ   0..0 'a'     cut[1] = min(0, cut[0]+1=0) = 0
  tâm 1 lẻ   1..1 'b'     cut[2] = min(1, cut[1]+1=1) = 1
  tâm 1 chẵn 1..2 'bb'    cut[3] = min(2, cut[1]+1=1) = 1  <- cải thiện
  tâm 1 chẵn 0..3 'abba'  cut[4] = min(3, cut[0]+1=0) = 0  <- cải thiện
  tâm 2 lẻ   2..2 'b'     cut[3] = min(1, cut[2]+1=2) = 1
  tâm 3 lẻ   3..3 'a'     cut[4] = min(0, cut[3]+1=2) = 0
  tâm 3 lẻ   2..4 'bab'   cut[5] = min(4, cut[2]+1=2) = 2  <- cải thiện
  tâm 4 lẻ   4..4 'b'     cut[5] = min(2, cut[4]+1=1) = 1  <- cải thiện
cut cuối = [-1, 0, 1, 1, 0, 1]


In [34]:
# Đáp án Bài 3 và Bài 4
def answer_expand(s: str) -> int:
    n = len(s)
    cut = [k - 1 for k in range(n + 1)]
    for c in range(n):
        for l, r in ((c, c), (c, c + 1)):
            while l >= 0 and r < n and s[l] == s[r]:
                if cut[l] + 1 < cut[r + 1]:
                    cut[r + 1] = cut[l] + 1
                l -= 1
                r += 1
    return cut[n]

def answer_dp_table(s: str) -> int:
    n = len(s)
    is_pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            is_pal[i][j] = s[i] == s[j] and (j - i < 2 or is_pal[i + 1][j - 1])
    dp = [0] * n
    for i in range(n):
        if is_pal[0][i]:
            continue                      # dp[i] = 0
        best = i
        for j in range(1, i + 1):
            if is_pal[j][i] and dp[j - 1] + 1 < best:
                best = dp[j - 1] + 1
        dp[i] = best
    return dp[n - 1]

for f in (answer_expand, answer_dp_table):
    for s, expected in tests:
        assert f(s) == expected, (f.__name__, s)
random.seed(132)
for _ in range(500):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 10)))
    assert answer_expand(s) == answer_dp_table(s) == min_cut_backtracking(s), s
print("Bài 3, Bài 4 PASS")

Bài 3, Bài 4 PASS


In [35]:
# Đáp án Bài 5 (LeetCode #1745) — O(n^2)
def answer_check_partitioning(s: str) -> bool:
    n = len(s)
    is_pal = build_is_pal(s)
    for i in range(1, n - 1):              # đoạn 1 = s[0..i-1]
        if not is_pal[0][i - 1]:
            continue
        for j in range(i + 1, n):          # đoạn 2 = s[i..j-1], đoạn 3 = s[j..n-1]
            if is_pal[i][j - 1] and is_pal[j][n - 1]:
                return True
    return False

def brute_check_partitioning(s: str) -> bool:
    return any(len(p) == 3 for p in palindrome_partitions(s))

assert answer_check_partitioning("abcbdd") is True
assert answer_check_partitioning("bcbddxy") is False
assert answer_check_partitioning("aaa") is True
assert answer_check_partitioning("abc") is True
random.seed(1745)
for _ in range(500):
    s = "".join(random.choice("ab") for _ in range(random.randint(3, 10)))
    assert answer_check_partitioning(s) == brute_check_partitioning(s), s
print("Bài 5 PASS")


# Đáp án Bài 6 (LeetCode #1278)
from itertools import combinations

def answer_palindrome_partition(s: str, k: int) -> int:
    n = len(s)
    cost = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            cost[i][j] = (s[i] != s[j]) + (cost[i + 1][j - 1] if j - i >= 2 else 0)
    INF = float("inf")
    f = [[INF] * (n + 1) for _ in range(k + 1)]
    f[0][0] = 0
    for m in range(1, k + 1):
        for i in range(m, n + 1):
            for j in range(m - 1, i):      # đoạn cuối s[j..i-1]
                if f[m - 1][j] + cost[j][i - 1] < f[m][i]:
                    f[m][i] = f[m - 1][j] + cost[j][i - 1]
    return f[k][n]

def brute_palindrome_partition(s: str, k: int) -> int:
    def fix(t):                            # số ký tự cần đổi để t thành palindrome
        return sum(t[x] != t[-1 - x] for x in range(len(t) // 2))
    best = len(s)
    for cuts in combinations(range(1, len(s)), k - 1):
        bounds = (0,) + cuts + (len(s),)
        best = min(best, sum(fix(s[bounds[x]:bounds[x + 1]]) for x in range(k)))
    return best

assert answer_palindrome_partition("abc", 2) == 1
assert answer_palindrome_partition("aabbc", 3) == 0
assert answer_palindrome_partition("leetcode", 8) == 0
random.seed(1278)
for _ in range(400):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 9)))
    k = random.randint(1, len(s))
    assert answer_palindrome_partition(s, k) == brute_palindrome_partition(s, k), (s, k)
print("Bài 6 PASS")

Bài 5 PASS
Bài 6 PASS


## 17. Template ngắn nên nhớ

**Mở rộng từ tâm + `cut` có lính canh** — câu thần chú: *"`cut[0] = -1`; mỗi tâm lẻ và chẵn, nới ra khi hai đầu bằng nhau; gặp palindrome `l..r` thì `cut[r+1] = min(cut[r+1], cut[l] + 1)`"*.

```python
class Solution:
    def minCut(self, s: str) -> int:
        n = len(s)
        cut = [k - 1 for k in range(n + 1)]
        for c in range(n):
            for l, r in ((c, c), (c, c + 1)):
                while l >= 0 and r < n and s[l] == s[r]:
                    cut[r + 1] = min(cut[r + 1], cut[l] + 1)
                    l -= 1
                    r += 1
        return cut[n]
```

**Khung DP "chia thành đoạn liên tiếp"** (dùng lại cho #139, #1278, #1745…):

```python
dp = [INF] * (n + 1); dp[0] = BASE          # dp[k]: đáp án cho s[0:k]
for i in range(1, n + 1):
    for j in range(i):                      # đoạn cuối s[j:i]
        if hop_le(j, i):                    # tra O(1) nhờ tính trước
            dp[i] = best(dp[i], dp[j] + gia_tri(j, i))
```

**Bảng palindrome "bóc vỏ":** `is_pal[i][j] = s[i] == s[j] and (j - i < 2 or is_pal[i+1][j-1])`, duyệt `i` giảm dần.

## 18. Checklist trước khi submit

- [ ] Mình trả về **số nhát cắt** (= số mảnh − 1), không phải số mảnh?
- [ ] Có lính canh `cut[0] = -1` (hoặc xử lý riêng trường hợp cả tiền tố là palindrome) chưa?
- [ ] Đã mở rộng **cả tâm lẻ `(c, c)` lẫn tâm chẵn `(c, c+1)`** chưa?
- [ ] Điều kiện `while` kiểm tra biên `l >= 0 and r < n` **trước** khi so `s[l] == s[r]`?
- [ ] Nếu dùng bảng `is_pal`: tạo bằng list comprehension (không `[[...]*n]*n`) và điền `i` **giảm dần**?
- [ ] Không còn kiểm tra palindrome bằng cắt chuỗi `O(n)` bên trong vòng lặp đôi (tránh `O(n³)`)?
- [ ] Không dùng đệ quy sâu tới `n = 2000` (tránh `RecursionError`)?
- [ ] Đã thử các ca: `"aab"`, `"a"`, `"ab"`, `"abaab"` (bẫy tham lam), `"abacc"`, `"aaaa"`, `"abcde"`?
- [ ] Độ phức tạp `O(n²)` thời gian, `O(n)` bộ nhớ — an toàn với `n = 2000`.